### 1. Phân tích P.E.A.S
* **P (Performance Measure):** Chỉ số hiệu suất dựa trên việc đưa bàn cờ về đúng trạng thái đích. Hiệu suất tối ưu được đánh giá bằng số bước di chuyển ít nhất để đạt mục tiêu (thuật toán BFS luôn đảm bảo tìm được đường đi ngắn nhất).
* **E (Environment):** Một lưới ô vuông 3x3 bao gồm các số từ 1 đến 8 và một ô trống (biểu diễn bằng số 0). Môi trường có thể quan sát toàn phần (Fully Observable) và mang tính tất định (Deterministic).
* **A (Actuators):** Các cơ cấu chấp hành thực hiện hành động di chuyển ô trống: LÊN (UP), XUỐNG (DOWN), TRÁI (LEFT), PHẢI (RIGHT) và DỪNG (STOP).
* **S (Sensors):** Cảm biến nhận diện trạng thái ma trận bàn cờ hiện tại, xác định vị trí ô trống, biết được các hành động hợp lệ tiếp theo và nhận biết khi nào đã đạt trạng thái đích (Goal).

### 2. Các thành phần của Agent theo kiến trúc Model-Based
* **State (Trạng thái hiện tại):** Là sự kết hợp giữa nhận thức (percept) ma trận 3x3 tại thời điểm hiện tại và bộ nhớ trong về không gian trạng thái.
* **Percept:** Dữ liệu nhận vào là một ma trận 3x3 thể hiện vị trí của các con số và ô trống hiện tại.
* **Model:**
    * **Hiểu biết về môi trường:** Agent hiểu rõ quy luật thay đổi của môi trường khi thực hiện hành động (ví dụ: hành động "LEFT" sẽ đổi chỗ ô trống với ô số đang nằm bên trái nó).
    * **Lưu trữ thông tin (Internal State):** Lưu trữ trạng thái đích (Goal), danh sách các trạng thái đã đi qua (Visited set) để tránh vòng lặp, hàng đợi (Queue) phục vụ cho thuật toán BFS, và Kế hoạch (Plan) lưu chuỗi hành động tối ưu cần thực hiện.
* **Rules (Tập luật và Lập luận IF):**
    * `IF` trạng thái bàn cờ hiện tại bằng với trạng thái đích (Goal) `THEN` Action = "STOP".
    * `IF` chưa đạt đích VÀ danh sách Plan đang trống `THEN` Kích hoạt lập luận tìm kiếm (dùng BFS duyệt theo chiều rộng) -> Cập nhật chuỗi hành động tìm được vào Plan -> Action = Plan.pop(0).
    * `IF` danh sách Plan đã có sẵn các bước đi `THEN` Action = Plan.pop(0).

### 3. Trạng thái môi trường (GOAL STATE)
Trạng thái đích đạt được khi tất cả các ô số được sắp xếp theo đúng thứ tự tăng dần và ô trống nằm ở góc dưới cùng bên phải.
Ví dụ với lưới 3x3:
+------+------+------+
|  1   |  2   |  3   |
+------+------+------+
|  4   |  5   |  6   |
+------+------+------+
|  7   |  8   |      |
+------+------+------+
*(0 hoặc khoảng trắng = blank tile)*

In [2]:
import copy
from collections import deque

def get_blank_pos(state):
    for r in range(3):
        for c in range(3):
            if state[r][c] == 0: return r, c

def get_heuristic(state, goal):
    dist = 0
    for r in range(3):
        for c in range(3):
            val = state[r][c]
            if val != 0:
                for gr in range(3):
                    for gc in range(3):
                        if goal[gr][gc] == val:
                            dist += abs(r - gr) + abs(c - gc)
    return dist

def get_successors(state):
    successors = []
    r, c = get_blank_pos(state)
    directions = {'UP': (-1, 0), 'DOWN': (1, 0), 'LEFT': (0, -1), 'RIGHT': (0, 1)}
    
    for action, (dr, dc) in directions.items():
        nr, nc = r + dr, c + dc
        if 0 <= nr < 3 and 0 <= nc < 3:
            new_state = copy.deepcopy(state)
            new_state[r][c], new_state[nr][nc] = new_state[nr][nc], new_state[r][c]
            successors.append((new_state, action))
    return successors

def bfs_search(start_state, goal_state):
    start_tuple = tuple(tuple(row) for row in start_state)
    queue = deque([(start_state, [])])
    visited = set()
    visited.add(start_tuple)

    while queue:
        current, path = queue.popleft()
        
        if current == goal_state:
            return path
            
        for next_state, action in get_successors(current):
            next_tuple = tuple(tuple(row) for row in next_state)
            if next_tuple not in visited:
                visited.add(next_tuple)
                queue.append((next_state, path + [action]))
    return []

class ModelBasedReflexAgent:
    def __init__(self, goal_state):
        self.state = None
        self.action = None
        self.model = {
            'goal': goal_state,
            'plan': [],
            'plan_count': 0
        }

    def agent_function(self, percept):
        self.state = percept
        
        if self.state == self.model['goal']:
            return 'STOP'
            
        if not self.model['plan']:
            self.model['plan'] = bfs_search(self.state, self.model['goal'])
            self.model['plan_count'] += 1
            
        if self.model['plan']:
            self.action = self.model['plan'].pop(0)
        else:
            self.action = 'STOP' 
            
        return self.action

def print_board(state, prev_state=None):
    changed_pos = []
    if prev_state:
        for r in range(3):
            for c in range(3):
                if state[r][c] != prev_state[r][c] and state[r][c] != 0:
                    changed_pos.append((r, c))

    print("+-----+-----+-----+")
    for r in range(3):
        row_str = "|"
        for c in range(3):
            val = state[r][c]
            if val == 0:
                row_str += "     |"
            else:
                if (r, c) in changed_pos:
                    row_str += f" [{val}] |"
                else:
                    row_str += f"  {val}  |"
        print(row_str)
        print("+-----+-----+-----+")

goal_state = [
    [1, 2, 3],
    [4, 5, 6],
    [7, 8, 0]
]

initial_percept = [
    [1, 2, 3],
    [5, 6, 0],
    [7, 8, 4]
]

agent = ModelBasedReflexAgent(goal_state)
current_percept = copy.deepcopy(initial_percept)
prev_percept = None
step = 0

print("=========================================================")
print(f"TRẠNG THÁI BAN ĐẦU (Step: {step} | Heuristic: {get_heuristic(current_percept, goal_state)})")
print("=========================================================")
print_board(current_percept)

while True:
    step += 1
    action = agent.agent_function(current_percept)
    
    if action == 'STOP':
        print("=========================================================")
        print(f"KẾT QUẢ: ĐÃ GIẢI XONG 8 PUZZLE TRONG {step - 1} BƯỚC!")
        print(f"Số lần agent phải lập Plan: {agent.model['plan_count']}")
        print("=========================================================")
        break
        
    prev_percept = copy.deepcopy(current_percept)
    r, c = get_blank_pos(current_percept)
    
    if action == 'UP': current_percept[r][c], current_percept[r-1][c] = current_percept[r-1][c], current_percept[r][c]
    elif action == 'DOWN': current_percept[r][c], current_percept[r+1][c] = current_percept[r+1][c], current_percept[r][c]
    elif action == 'LEFT': current_percept[r][c], current_percept[r][c-1] = current_percept[r][c-1], current_percept[r][c]
    elif action == 'RIGHT': current_percept[r][c], current_percept[r][c+1] = current_percept[r][c+1], current_percept[r][c]
    
    print("---------------------------------------------------------")
    print(f"Step: {step:02d} | Action: {action} | Heuristic: {get_heuristic(current_percept, goal_state)}")
    print_board(current_percept, prev_percept)

TRẠNG THÁI BAN ĐẦU (Step: 0 | Heuristic: 5)
+-----+-----+-----+
|  1  |  2  |  3  |
+-----+-----+-----+
|  5  |  6  |     |
+-----+-----+-----+
|  7  |  8  |  4  |
+-----+-----+-----+
---------------------------------------------------------
Step: 01 | Action: LEFT | Heuristic: 4
+-----+-----+-----+
|  1  |  2  |  3  |
+-----+-----+-----+
|  5  |     | [6] |
+-----+-----+-----+
|  7  |  8  |  4  |
+-----+-----+-----+
---------------------------------------------------------
Step: 02 | Action: LEFT | Heuristic: 3
+-----+-----+-----+
|  1  |  2  |  3  |
+-----+-----+-----+
|     | [5] |  6  |
+-----+-----+-----+
|  7  |  8  |  4  |
+-----+-----+-----+
---------------------------------------------------------
Step: 03 | Action: DOWN | Heuristic: 4
+-----+-----+-----+
|  1  |  2  |  3  |
+-----+-----+-----+
| [7] |  5  |  6  |
+-----+-----+-----+
|     |  8  |  4  |
+-----+-----+-----+
---------------------------------------------------------
Step: 04 | Action: RIGHT | Heuristic: 5
+-----+